# DyBAS-Net trên Electricity (321 biến) và Traffic (862 biến), L = 720

**Cách dùng:** *Runtime → Change runtime type → T4 GPU* → *Runtime → Run all*. Giữ tab mở đến khi ô cuối cùng tải về file `dybas_results.zip`, rồi gửi file đó lại.

Mọi lần chạy cùng pipeline với 5 bộ đã có (4 horizon × 3 seed, siêu tham số chính thức của từng mô hình, `scripts/run_suite.py`). Kết quả được ghi ngay vào Google Drive (`MyDrive/DyBAS-Net/results/`) sau mỗi lần chạy. **Nếu phiên bị ngắt hoặc hết quota GPU:** mở lại notebook và *Run all* lần nữa (có thể vào hôm sau) — những lần chạy đã xong được bỏ qua, notebook chạy tiếp từ chỗ dở.

| Giai đoạn | Nội dung | Số lần chạy mỗi bộ |
|---|---|---|
| 1 | DyBAS-Net (D = 64, p1 = p2 = 0.1) và TimeBase\* | 24 |
| 2 | CycleNet/Linear, CycleNet/MLP | 24 |
| 3 | Lưới DropFusion của DyBAS-Net: p1, p2 ∈ {0.1, 0.3, 0.5}, seed 2021, chọn theo val MSE; cặp tốt nhất chạy thêm 2 seed | ≤ 40 |
| 4 | TQNet (nặng nhất) | 12 |
| Cuối | Bảng tổng hợp + nén kết quả, tự tải về | — |

In [ ]:
import torch
assert torch.cuda.is_available(), 'Chưa bật GPU: Runtime → Change runtime type → T4 GPU, rồi Run all lại.'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUT = '/content/drive/MyDrive/DyBAS-Net/results'
os.makedirs(OUT, exist_ok=True)
print('Kết quả lưu tại', OUT)

In [ ]:
%cd /content
![ -d DyBAS-Net ] || git clone -q -b exp/dytimebase https://github.com/longdanghoclaptrinh/DyBAS-Net.git
%cd /content/DyBAS-Net
!git pull -q; git log -1 --format='%h %s'

In [ ]:
# Dữ liệu: bản benchmark của THUML trên Hugging Face (cùng nguồn với ETT/Weather trong data/README.md).
from pathlib import Path
from urllib.request import urlretrieve
import pandas as pd

EXPECTED = {'electricity.csv': (26304, 321), 'traffic.csv': (17544, 862)}
base = 'https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/main'
for local, remote in {'electricity.csv': 'electricity/electricity.csv', 'traffic.csv': 'traffic/traffic.csv'}.items():
    dst = Path('data') / local
    if not dst.exists():
        urlretrieve(f'{base}/{remote}?download=true', dst)
    df = pd.read_csv(dst)
    shape = (len(df), df.shape[1] - 1)
    assert shape == EXPECTED[local], (local, shape)
    print(local, shape[0], 'dòng,', shape[1], 'biến')

In [ ]:
import json, statistics, subprocess, sys, time

DATASETS = ('Electricity', 'Traffic')
H, SEEDS = (96, 192, 336, 720), (2021, 2022, 2023)

def load(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else {}

def run(dataset, variants, results, seeds=SEEDS):
    """Chạy scripts/run_suite.py; bỏ qua lần chạy đã có trong file kết quả trên Drive."""
    cmd = [sys.executable, '-u', 'scripts/run_suite.py', '--dataset', dataset, '--seq_len', '720', '--device', 'cuda',
           '--variants', *variants, '--seeds', *map(str, seeds), '--results', results]
    t = time.time()
    with open(f'{OUT}/log_{dataset}.txt', 'a') as log:
        code = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT).returncode
    print(f'{dataset} {" ".join(variants)}: exit {code}, {(time.time() - t) / 60:.1f} phút')
    if code:
        print(open(f'{OUT}/log_{dataset}.txt').read()[-3000:])
        raise RuntimeError(f'Lỗi khi chạy {dataset}; xem log ở trên')

def suite(ds):
    return f'{OUT}/suite_{ds}_L720.json'

def grid(ds):
    return f'{OUT}/grid_{ds}_L720.json'

def progress():
    for ds in DATASETS:
        count = {}
        for r in {**load(suite(ds)), **load(grid(ds))}.values():
            count[r['variant']] = count.get(r['variant'], 0) + 1
        print(f'{ds}:', ', '.join(f'{v} {n}' for v, n in sorted(count.items())) or 'chưa có')

## Giai đoạn 1: DyBAS-Net và TimeBase\*

In [ ]:
for ds in DATASETS:
    run(ds, ['dytbase_off', 'timebase_off'], suite(ds))
progress()

## Giai đoạn 2: CycleNet/Linear và CycleNet/MLP

In [ ]:
for ds in DATASETS:
    run(ds, ['cyclenet_off', 'cyclenet_mlp_off'], suite(ds))
progress()

## Giai đoạn 3: lưới DropFusion của DyBAS-Net

D = 64 cố định. p1, p2 ∈ {0.1, 0.3, 0.5} (lưới thô hơn ETT vì chi phí). Chọn cặp có val MSE trung bình 4 horizon thấp nhất ở seed 2021 — không nhìn test. Cặp (0.1, 0.1) chính là `dytbase_off` của giai đoạn 1. Nếu cặp tốt nhất khác (0.1, 0.1), chạy thêm seed 2022, 2023 cho cặp đó.

In [ ]:
P = (0.1, 0.3, 0.5)
CHOSEN = {}
for ds in DATASETS:
    others = [f'dytbase_d64_p{a:g}_{b:g}' for a in P for b in P if (a, b) != (0.1, 0.1)]
    run(ds, others, grid(ds), seeds=(2021,))
    runs = {**load(suite(ds)), **load(grid(ds))}
    val = {}
    for v in ['dytbase_off'] + others:
        val[v] = statistics.mean(runs[f'{v}_S{h}_seed2021']['best_val'] for h in H)
    best = min(val, key=val.get)
    CHOSEN[ds] = best
    print(ds, 'val MSE TB (seed 2021):', ', '.join(f'{v.replace("dytbase_", "")} {x:.4f}' for v, x in sorted(val.items(), key=lambda kv: kv[1])))
    print(ds, '→ chọn', best)
    if best != 'dytbase_off':
        run(ds, [best], grid(ds))
json.dump(CHOSEN, open(f'{OUT}/chosen.json', 'w'), indent=1)
progress()

## Giai đoạn 4: TQNet

In [ ]:
for ds in DATASETS:
    run(ds, ['tqnet_off'], suite(ds))
progress()

## Tổng hợp và tải kết quả

MSE/MAE trung bình 4 horizon × 3 seed (tự chạy) và số công bố trong Table 10 của TimeBase (cùng L = 720).

In [ ]:
CHOSEN = load(f'{OUT}/chosen.json')
pub = json.load(open('results/published_timebase_table10.json'))['results']
lines = ['| Mô hình | ' + ' | '.join(f'{ds} MSE | {ds} MAE' for ds in DATASETS) + ' |', '|---|' + '---:|' * 4]
OURS = {'DyBAS-Net (p1 = p2 = 0.1)': 'dytbase_off', 'DyBAS-Net (p chọn theo val)': None, 'TimeBase*': 'timebase_off',
        'CycleNet/Linear': 'cyclenet_off', 'CycleNet/MLP': 'cyclenet_mlp_off', 'TQNet': 'tqnet_off'}
for label, v in OURS.items():
    cells = []
    for ds in DATASETS:
        runs = {**load(suite(ds)), **load(grid(ds))}
        name = v or CHOSEN.get(ds)
        keys = [f'{name}_S{h}_seed{s}' for h in H for s in SEEDS]
        if name and all(k in runs for k in keys):
            cells += [f"{statistics.mean(runs[k][m] for k in keys):.4f}" for m in ('mse', 'mae')]
        else:
            cells += ['chưa đủ', 'chưa đủ']
    lines.append(f'| {label} | ' + ' | '.join(cells) + ' |')
for m in ('TimeBase', 'SparseTSF', 'FITS', 'PatchTST', 'iTransformer', 'TimeMixer', 'DLinear'):
    cells = []
    for ds in DATASETS:
        cells += [f"{statistics.mean(pub[ds][m][str(h)][k] for h in H):.4f}" for k in ('mse', 'mae')]
    lines.append(f'| {m} (công bố) | ' + ' | '.join(cells) + ' |')
summary = '\n'.join(lines)
Path(f'{OUT}/summary.md').write_text(f'DyBAS-Net chọn theo val: {CHOSEN}\n\n{summary}\n')
from IPython.display import Markdown, display
display(Markdown(summary))
progress()

In [ ]:
import shutil
shutil.make_archive('/content/dybas_results', 'zip', OUT)
from google.colab import files
files.download('/content/dybas_results.zip')
print('Đã nén', OUT, '→ dybas_results.zip (bản gốc vẫn ở Drive). Gửi file này lại.')